# Análisis de Grupos con GroupBy y Funciones de Agregación en Pandas

## 🎯 Objetivos
Al finalizar este notebook, serás capaz de:
- Implementar la estrategia **Split-Apply-Combine** para análisis de datos.
- Utilizar el método `groupby()` para segmentar información basada en categorías.
- Aplicar funciones de agregación personalizadas mediante el método `agg()`.
- Generar resúmenes estadísticos complejos para la toma de decisiones.

## 📖 Introducción

En el análisis de datos, rara vez nos interesa el promedio global de una métrica. Lo que realmente aporta valor es la **comparación entre segmentos**. Por ejemplo, no es lo mismo conocer el precio promedio de todos los autos que conocer el precio promedio *por marca* o *por tipo de vehículo*.

Pandas implementa este flujo de trabajo a través de la potencia de `groupby()`, permitiéndonos transformar millones de filas en resúmenes accionables.

In [ ]:
import pandas as pd
from pathlib import Path

# Configuración de rutas
DATA_PATH = Path(".")

## 1. La Estrategia Split-Apply-Combine

### 💡 Intuición
Esta es la base de casi todas las operaciones de agregación en ciencia de datos. El proceso se divide en tres etapas claras:

1. **Split (Dividir)**: El conjunto de datos se divide en grupos basados en una clave (ej. `Manufacturer`).
2. **Apply (Aplicar)**: Se aplica una función a cada grupo de forma independiente (ej. `mean()`).
3. **Combine (Combinar)**: Los resultados de cada grupo se vuelven a unir en una estructura de datos final.

### 🛠️ Diagrama de Concepto
```
 [ DataFrame Original ]
           |
           v
      ( SPLIT )  ------>  Segmentar por Clave (ej. "Tipo de Vehículo")
           |
           +------------> [ Grupo: Car ] ---- ( APPLY: mean ) ----+ 
           +------------> [ Grupo: Pass ] --- ( APPLY: mean ) ----+ --> ( COMBINE )
           |
           v
   [ DataFrame Resumen ]
```

In [ ]:
# Carga de datos
df_cars = pd.read_csv(DATA_PATH / "Car_sales.csv")

# Seleccionamos columnas relevantes para el análisis
cols = ['Manufacturer', 'Sales_in_thousands', 'Vehicle_type', 
        'Price_in_thousands', 'Engine_size', 'Horsepower']
df_cars = df_cars[cols]

display(df_cars.head())

## 2. El método `groupby()`

### 🛠️ Implementación
Cuando llamamos a `groupby()`, Pandas no calcula nada inmediatamente. Crea un objeto `DataFrameGroupBy`, que es básicamente una "promesa" de que los datos están organizados y listos para una operación.

In [ ]:
# Crear el objeto groupby
grouped_manufacturer = df_cars.groupby('Manufacturer')

print(f"Tipo de objeto creado: {type(grouped_manufacturer)}")

# Acceder a un grupo específico
ford_data = grouped_manufacturer.get_group('Ford')
print("\n--- Datos solo de Ford ---")
display(ford_data.head())

In [ ]:
# Operación directa: Promedio de ventas por tipo de vehículo
avg_sales = df_cars.groupby('Vehicle_type')['Sales_in_thousands'].mean()

print("--- Ventas Promedio por Tipo de Vehículo ---")
display(avg_sales)

## 3. El método `agg()` (Agregaciones Avanzadas)

### 💡 Intuición
A veces, una sola función (como `mean`) no es suficiente. Queremos saber el total, el promedio y el valor máximo al mismo tiempo, o aplicar diferentes funciones a diferentes columnas.

### 🛠️ Diagrama de `agg()`
```
 [ DataFrame ]  ---> [ .agg({ 'Ventas': 'sum', 'Precio': 'mean' }) ]
                                    |
                                    v
 [ Resultado ] ---> { 'Ventas': Total_Global, 'Precio': Promedio_Global }
```

In [ ]:
# 1. Múltiples funciones para todas las columnas numéricas
global_summary = df_cars.agg(['sum', 'mean', 'max'])

print("--- Resumen Global (Suma, Promedio, Máximo) ---")
display(global_summary)

In [ ]:
# 2. Funciones específicas por columna usando un diccionario
custom_summary = df_cars.agg({
    'Sales_in_thousands': ['sum', 'mean'],
    'Price_in_thousands': ['max', 'min'],
    'Horsepower': 'mean'
})

print("--- Resumen Personalizado por Columna ---")
display(custom_summary)

## 4. Combinando GroupBy y Agg

El verdadero poder surge cuando combinamos ambos. Podemos segmentar los datos y luego aplicar agregaciones específicas a cada segmento.

In [ ]:
# Análisis por Marca: Ventas totales y Precio promedio
brand_analysis = df_cars.groupby('Manufacturer').agg({
    'Sales_in_thousands': 'sum',
    'Price_in_thousands': 'mean'
}).rename(columns={
    'Sales_in_thousands': 'Ventas_Totales',
    'Price_in_thousands': 'Precio_Promedio'
})

print("--- Análisis de Marcas ---")
display(brand_analysis.sort_values(by='Ventas_Totales', ascending=False).head(10))

## 📝 Ejercicios de Práctica

**Ejercicio 1**: Utilizando el dataset `Car_sales.csv`, encuentra la potencia promedio (`Horsepower`) y el tamaño de motor promedio (`Engine_size`) para cada tipo de vehículo (`Vehicle_type`).

**Ejercicio 2**: Carga el dataset `vgsales.csv`. Agrupa los datos por plataforma (`Platform`) y calcula la cantidad total de juegos vendidos (`Global_Sales`) y el número de juegos lanzados (usa la función `count`).

In [ ]:
# Solución Ejercicio 1
# TODO: Implementar aquí
pass

In [ ]:
# Solución Ejercicio 2
# TODO: Implementar aquí
pass

## 📋 Resumen Rápido

| Herramienta | Función Principal | Ejemplo de uso |
| :--- | :--- | :--- |
| `groupby()` | Segmentar datos por categorías | `df.groupby('Categoria')` |
| `.mean()`, `.sum()` | Agregación básica rápida | `grouped.mean()` |
| `.agg()` | Agregaciones múltiples y personalizadas | `df.agg({'Col': 'max'})` |
| `.get_group()` | Extraer un segmento específico | `grouped.get_group('Ford')` |